# 00 — Data Exploration

**Goal:** Understand the distribution of harm verticals in BeaverTails and RealToxicityPrompts,
quantify the sampling challenge, and produce a *sampling difficulty table* showing how many
labeled items are required per category to achieve a ±1pp prevalence CI.

**Key output:** Evidence that naive random sampling fails for rare verticals, motivating the
stratified and risk-score-stratified designs in `02_sampling_design.ipynb`.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

from src.sampling import StratifiedHarmSampler

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED = 42
np.random.seed(SEED)

print('Environment ready.')

## 1. Load BeaverTails

BeaverTails contains 333,963 QA pairs with binary safety labels and 14 harm category flags.
We treat the full dataset as our proxy "platform corpus" with known ground truth — which lets
us evaluate prevalence estimators against actual labels in later notebooks.

In [ ]:
try:
    from datasets import load_dataset
    print('Loading BeaverTails from HuggingFace...')
    ds = load_dataset('PKU-Alignment/BeaverTails', split='330k_train')
    df = ds.to_pandas()
    print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
except Exception as e:
    print(f'Could not load BeaverTails ({e}). Generating synthetic stand-in.')
    # Synthetic stand-in that mirrors BeaverTails schema and approximate prevalence rates
    from src.simulation import SimulationConfig, generate_corpus
    rng = np.random.default_rng(SEED)
    n = 50_000
    harm_prevalences = {
        'hate_speech': 0.042,
        'violence': 0.038,
        'self_harm': 0.021,
        'illegal_activity': 0.065,
        'animal_abuse': 0.008,
        'child_abuse': 0.004,
        'controversial_topics': 0.112,
        'discrimination': 0.031,
        'financial_crime': 0.019,
        'non_violent_unethical': 0.089,
        'privacy_violation': 0.022,
        'sexually_explicit': 0.055,
        'terrorism': 0.011,
        'weapons': 0.016,
    }
    data = {'is_safe': rng.binomial(1, 0.55, n).astype(bool)}
    for col, prev in harm_prevalences.items():
        data[col] = rng.binomial(1, prev, n).astype(bool)
    df = pd.DataFrame(data)
    print(f'Synthetic corpus: {len(df):,} rows')

df.head(3)

## 2. Label Distribution

How prevalent is each harm vertical? This drives the entire sampling design.

In [ ]:
# Identify harm category columns
harm_cols = [c for c in df.columns if c not in ('prompt', 'response', 'is_safe', 'category')]
print(f'Harm verticals detected: {harm_cols}')

# Compute prevalence per vertical
prevalence = df[harm_cols].mean().sort_values(ascending=False)
print('\nPrevalence per harm vertical:')
print(prevalence.map(lambda x: f'{x:.3%}').to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#d62728' if v > 0.05 else '#ff7f0e' if v > 0.01 else '#1f77b4'
          for v in prevalence.values]
bars = ax.barh(prevalence.index, prevalence.values * 100, color=colors)

ax.set_xlabel('Prevalence (%)')
ax.set_title('Harm Vertical Prevalence in BeaverTails\n'
             '(red = >5%, orange = 1–5%, blue = <1%)', fontsize=13)
ax.axvline(1.0, color='black', linestyle='--', alpha=0.5, label='1% threshold')
ax.axvline(5.0, color='red', linestyle='--', alpha=0.5, label='5% threshold')
ax.legend()

for bar, val in zip(bars, prevalence.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{val:.2%}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/00_prevalence_by_vertical.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig saved.')

## 3. Why Naive Random Sampling Fails

For verticals with < 1% prevalence, a simple random sample returns so few positive
examples that statistical estimates are unreliable. The simulation below shows the
expected positive count and CI behavior under SRS.

In [ ]:
sampler = StratifiedHarmSampler(confidence_level=0.95)
CORPUS_SIZE = 10_000_000  # Hypothetical platform scale
TARGET_CI = 0.01          # ±1pp CI

rows = []
for category, prev in prevalence.items():
    n_srs = sampler.minimum_sample_size_srs(prev, TARGET_CI)
    expected_pos_srs = n_srs * prev
    rows.append({
        'harm_vertical': category,
        'true_prevalence': prev,
        'n_srs_required': n_srs,
        'expected_positives_in_srs': expected_pos_srs,
        'labeling_cost_usd': n_srs * 0.001,  # $0.001 per LLM label
        'feasible': expected_pos_srs >= 10,
    })

difficulty = pd.DataFrame(rows).sort_values('true_prevalence')
difficulty['true_prevalence_pct'] = difficulty['true_prevalence'].map(lambda x: f'{x:.3%}')
difficulty['expected_positives_in_srs'] = difficulty['expected_positives_in_srs'].map(lambda x: f'{x:.1f}')
difficulty['n_srs_required'] = difficulty['n_srs_required'].map(lambda x: f'{x:,}')
difficulty['labeling_cost_usd'] = difficulty['labeling_cost_usd'].map(lambda x: f'${x:,.2f}')

print(f'Sampling Difficulty Table — Target CI: ±{TARGET_CI:.0%}, Corpus: {CORPUS_SIZE:,}')
print(difficulty[['harm_vertical', 'true_prevalence_pct', 'n_srs_required',
                   'expected_positives_in_srs', 'labeling_cost_usd', 'feasible']].to_string(index=False))

In [ ]:
# Visualize: required sample size vs prevalence rate
prev_range = np.logspace(-4, -1, 200)  # 0.01% to 10%
n_required = [sampler.minimum_sample_size_srs(p, TARGET_CI) for p in prev_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(prev_range * 100, n_required, 'b-', lw=2)
ax.axhline(10_000, color='orange', linestyle='--', label='10K label budget')
ax.axhline(100_000, color='red', linestyle='--', label='100K label budget')

# Annotate actual verticals
for _, row in prevalence.items():
    n = sampler.minimum_sample_size_srs(row, TARGET_CI) if row > 1e-5 else None

ax.set_xlabel('True Prevalence (%)')
ax.set_ylabel('SRS Sample Size Required for ±1pp CI')
ax.set_title('Sample Size vs Prevalence (SRS)\nShaded region = infeasible under 10K budget')
ax.fill_between(prev_range * 100, 10_000, max(n_required), alpha=0.1, color='red',
                label='Infeasible (<10K budget)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/00_srs_sample_size_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Between Harm Verticals

Understanding co-occurrence helps design multi-label classifiers and informs whether
a single oversampled item can serve multiple measurement objectives.

In [ ]:
corr = df[harm_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-0.5, vmax=0.5, ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Pairwise Correlation Between Harm Verticals', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/00_harm_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# High correlations suggest harm co-occurrence — relevant for actor-level measurement
strong_pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.15:
            strong_pairs.append((corr.columns[i], corr.columns[j], r))

if strong_pairs:
    print('Strong vertical correlations (|r| > 0.15):')
    for a, b, r in sorted(strong_pairs, key=lambda x: -abs(x[2])):
        print(f'  {a} × {b}: r={r:.3f}')

## 5. Key Takeaways

| Finding | Implication |
|---------|------------|
| Most harm verticals have < 5% prevalence | SRS is expensive and often infeasible |
| Child abuse / terrorism < 0.5% | Require risk-score stratification for any detection signal |
| High vertical correlations | Multi-label annotation amortizes labeling cost |
| Rare verticals: expected SRS positives < 5 | CI inference is unreliable without stratification |

**Next step:** `01_llm_classifier.ipynb` — run the LLM-based detection system on a stratified subset.